# Rebuilt liquidity evidence: review companion

Four-market historical activity, shared observation dates and source-quality limits. This is a retrospective descriptive study, not calibrated execution capacity or historical point-in-time proof.

Set `LIQUIDITY_REVIEW_RUN` to the external reviewed run directory and `LIQUIDITY_REVIEW_OUTPUT` to its review directory. No vendor rows, securities or private paths are embedded in this notebook. Rates are [ECB reference rates](https://www.ecb.europa.eu/stats/policy_and_exchange_rates/euro_reference_exchange_rates/html/index.en.html), using only dates strictly earlier than the observation. US turnover is a close-times-volume proxy; Japanese caps use exploratory Yahoo share history.

In [ ]:
import os, json, hashlib
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
run = Path(os.environ["LIQUIDITY_REVIEW_RUN"])
review_dir = Path(os.environ["LIQUIDITY_REVIEW_OUTPUT"])
summary_bytes = (run / "summary.proposed.json").read_bytes()
summary = json.loads(summary_bytes)
review = json.loads((review_dir / "aggregate-review.json").read_text())
assert hashlib.sha256(summary_bytes).hexdigest() == review["reviewed_summary_sha256"]
assert review["count_and_reason_conservation"] and review["strict_prior_fx_checked"]
display(pd.DataFrame([{k: review[k] for k in ["aggregate_cells_reconciled", "security_day_observations_checked", "full_raw_warehouse_audited", "actual_execution_calibrated"]}]))

The review counts entries across publication views that overlap; it does not count unique raw rows independently audited.

## 1. Common dates and sample grain

Latest market snapshots differ in date. The common snapshot is one shared observation date. Historical windows pool equally weighted security-days; stocks can cross buckets over time, so distinct-stock counts are not additive.

In [ ]:
periods = summary["periods"]
display(pd.DataFrame([{k:p[k] for k in ["period","common_start","common_end","common_dates","missing_markets"]} for p in periods]))
common = next(p for p in periods if p["period"] == "common snapshot")
for p in periods:
    for m in p["markets"]:
        assert sum(b["observations"] for b in m["buckets"]) == m["eligible_observations"]
        if m["aggregation"] == "snapshot":
            assert sum(b["count"] for b in m["buckets"]) == m["eligible_stocks"]
display(pd.DataFrame([{k:m[k] for k in ["market","as_of","eligible_stocks","sub_100m_count"]} for m in common["markets"]]))

## 2. Turnover distributions on one date

Compare the same size buckets. Empty buckets stay missing rather than becoming zero. Logarithmic values show scale; different source universes and turnover proxies remain material limitations.

In [ ]:
rows = [{"market":m["market"], "bucket":b["label"], "ADV20":b["median_usd"], "median20":b["median20_median_usd"], "stocks":b["count"]} for m in common["markets"] for b in m["buckets"]]
table = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(10,5), constrained_layout=True)
table.pivot(index="bucket", columns="market", values="ADV20").reindex([b["label"] for b in common["markets"][0]["buckets"]]).plot.bar(ax=ax, logy=True)
ax.set(title=f"Prior 20-session mean turnover — common date {common['common_end']}", ylabel="Group median, USD reference FX (log scale)", xlabel="Prior-session market-cap bucket")
ax.set_xticklabels([label.get_text().replace("$", "") for label in ax.get_xticklabels()])
ax.tick_params(axis="x", rotation=15)
ax.grid(axis="y", alpha=.2)
fig.savefig(review_dir / "common-liquidity.png", dpi=150)
plt.show()
display(table)

## 3. Mean versus median daily turnover

These are distinct rolling statistics for each stock. Compare their group medians without renaming one as the other. Small-group stock counts must accompany any interpretation.

In [ ]:
selected = table.loc[table.bucket == "$10m-$100m"].set_index("market")
fig, ax = plt.subplots(figsize=(8,4), constrained_layout=True)
selected[["ADV20","median20"]].plot.bar(ax=ax, logy=True, color=["#377eb8","#d08240"])
ax.set(title="Same size bucket: USD 10m–100m", ylabel="Group median turnover, USD (log scale)", xlabel="Market")
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y", alpha=.2)
fig.savefig(review_dir / "mean-median-comparison.png", dpi=150)
plt.show()
display(selected)

## 4. Source-panel exclusions

Denominator: all normalized source rows from November 2019 through each source’s end date. These rates describe source/feature coverage, not probabilities of executable trades. Exclusion categories form an ordered, mutually exclusive ledger.

In [ ]:
profile = pd.DataFrame({m:p["exclusions"] for m,p in summary["source_profiles"].items()}).fillna(0).T
for m,p in summary["source_profiles"].items():
    assert profile.loc[m].sum() == p["source_rows"]
shares = profile.div(profile.sum(axis=1),axis=0)
fig, ax = plt.subplots(figsize=(10,5), constrained_layout=True)
labels = {"included":"Usable", "insufficient_history":"Short history", "missing_market_sessions":"Missing sessions", "missing_turnover":"Missing turnover", "prior_source_ineligible":"Prior source rule", "missing_prior_cap":"Missing prior cap", "missing_or_stale_fx":"FX missing/stale", "numeric_overflow":"Numeric overflow", "sparse_non_market_date":"Sparse source dates"}
colors = {"included":"#26846b", "insufficient_history":"#a6bddb", "missing_market_sessions":"#ddaa55", "missing_turnover":"#bc80bd", "prior_source_ineligible":"#b3b3b3", "missing_prior_cap":"#d77762", "missing_or_stale_fx":"#8c6bb1", "numeric_overflow":"#333333", "sparse_non_market_date":"#5577aa"}
shares.rename(columns=labels).plot.barh(stacked=True,ax=ax,color=[colors[k] for k in shares.columns])
ax.set(title="Coverage of normalized source panels", xlabel="Fraction of source rows", ylabel="Market", xlim=(0,1))
ax.legend(loc="lower left",bbox_to_anchor=(0,1.15),ncol=3,fontsize=8)
fig.savefig(review_dir / "source-coverage.png",dpi=150)
plt.show()
display(profile.astype(int))

## 5. Concentrated execution-minute gap

This audit covers saved sampled selection keys, not all exchange records. Identical missing minutes across sampled Shenzhen securities locate the issue by venue/time; they do not identify the upstream collection or conversion cause.

In [ ]:
gaps = json.loads((review_dir / "minute-gap-audit.proposed.json").read_text())
display(pd.DataFrame([dict(day=d["day"], **{k:v[k] for k in ["venue","observed_stocks","affected_stocks","expected_minutes","observed_minutes","missing_minutes"]}) for d in gaps["days"] for v in d["venues"]]))
display(pd.DataFrame([dict(day=d["day"],venue=v["venue"],**block) for d in gaps["days"] for v in d["venues"] for block in v["blocks"]]))

## What remains open

Extend the US source beyond October 2025 and the HK/JP sources beyond their present endpoints. Validate historical share classes, Japanese share counts, instrument-universe completeness and original data vintages. Investigate raw order/quote semantics and obtain real parent/child order and fill records before claiming calibrated executable capacity. The 2026 window explicitly excludes the unavailable US market.